In [ ]:
import math

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def sigmoid_derivative(sig_x):
    return sig_x * (1 - sig_x)

def binary_cross_entropy(y_hat, y):
    return - (y * math.log(y_hat) + (1 - y) * math.log(1 - y_hat))

class TwoLayerNeuralNetwork:
    def __init__(self, learning_rate=0.1):
        # Layer 1 weights and biases
        self.W_a, self.W_b, self.bias_b_a = 0.5, -0.3, 0.1
        self.W_c, self.W_d, self.bias_b_b = -0.7, 0.8, -0.2
        
        # Layer 2 weights and biases
        self.W_e, self.W_f, self.bias_b_c = 0.3, -0.5, 0.05
        self.W_h, self.W_i, self.bias_b_d = 0.2, -0.4, 0.1
        
        # Output layer weights and bias
        self.W_z, self.bias_b = -0.6, 0.2
        
        self.lr = learning_rate

    def forward(self, x1, x2):
        # Layer 1
        self.Z_a = self.W_a * x1 + self.bias_b_a
        self.h_a = sigmoid(self.Z_a)
        
        self.Z_b = self.W_b * x1 + self.bias_b_b
        self.h_b = sigmoid(self.Z_b)
        
        self.Z_c = self.W_c * x2 + self.bias_b_c
        self.h_c = sigmoid(self.Z_c)
        
        self.Z_d = self.W_d * x2 + self.bias_b_d
        self.h_d = sigmoid(self.Z_d)

        # Layer 2
        self.Z_e = self.W_e * self.h_a + self.W_f * self.h_c + self.bias_b_c
        self.h_e = sigmoid(self.Z_e)
        
        self.Z_f = self.W_h * self.h_b + self.W_i * self.h_d + self.bias_b_d
        self.h_f = sigmoid(self.Z_f)

        # Output layer
        self.z = self.W_z * self.h_e + self.bias_b
        self.y_hat = sigmoid(self.z)
        return self.y_hat

    def backward(self, x1, x2, y):
        # Output layer gradients
        dL_dyhat = self.y_hat - y
        dz = dL_dyhat  # because sigmoid + BCE simplifies

        self.dL_dW_z = dz * self.h_e
        self.dL_dbias_b = dz

        # Layer 2 gradients
        dh_e = dz * self.W_z * sigmoid_derivative(self.h_e)
        self.dL_dW_e = dh_e * self.h_a
        self.dL_dW_f = dh_e * self.h_c
        self.dL_dbias_b_c = dh_e

        dh_f = 0  # Not connected to output in this architecture
        self.dL_dW_h = dh_f * self.h_b
        self.dL_dW_i = dh_f * self.h_d
        self.dL_dbias_b_d = dh_f

        # Layer 1 gradients
        dh_a = dh_e * self.W_e * sigmoid_derivative(self.h_a)
        self.dL_dW_a = dh_a * x1
        self.dL_dbias_b_a = dh_a

        dh_b = dh_f * self.W_h * sigmoid_derivative(self.h_b)
        self.dL_dW_b = dh_b * x1
        self.dL_dbias_b_b = dh_b

        dh_c = dh_e * self.W_f * sigmoid_derivative(self.h_c)
        self.dL_dW_c = dh_c * x2
        self.dL_dbias_b_c += dh_c  # Note: += because bias_b_c is used in multiple places

        dh_d = dh_f * self.W_i * sigmoid_derivative(self.h_d)
        self.dL_dW_d = dh_d * x2
        self.dL_dbias_b_d += dh_d

    def update(self):
        # Output layer update
        self.W_z -= self.lr * self.dL_dW_z
        self.bias_b -= self.lr * self.dL_dbias_b

        # Layer 2 update
        self.W_e -= self.lr * self.dL_dW_e
        self.W_f -= self.lr * self.dL_dW_f
        self.W_h -= self.lr * self.dL_dW_h
        self.W_i -= self.lr * self.dL_dW_i
        self.bias_b_c -= self.lr * self.dL_dbias_b_c
        self.bias_b_d -= self.lr * self.dL_dbias_b_d

        # Layer 1 update
        self.W_a -= self.lr * self.dL_dW_a
        self.W_b -= self.lr * self.dL_dW_b
        self.W_c -= self.lr * self.dL_dW_c
        self.W_d -= self.lr * self.dL_dW_d
        self.bias_b_a -= self.lr * self.dL_dbias_b_a
        self.bias_b_b -= self.lr * self.dL_dbias_b_b

    def train(self, X, Y, epochs=1000):
        for epoch in range(epochs):
            total_loss = 0
            for (x1, x2), y in zip(X, Y):
                y_hat = self.forward(x1, x2)
                loss = binary_cross_entropy(y_hat, y)
                total_loss += loss
                self.backward(x1, x2, y)
                self.update()
            if (epoch+1) % 1000 == 0:
                print(f"Epoch {epoch+1:3d}, Loss: {total_loss:.4f}")

    def predict(self, x1, x2):
        y_hat = self.forward(x1, x2)
        return 1 if y_hat >= 0.5 else 0

    def test(self, X_test, Y_test):
        correct = 0
        for (x1, x2), y in zip(X_test, Y_test):
            pred = self.predict(x1, x2)
            print(f"Input: ({x1}, {x2}) → Predicted: {pred}, Actual: {y}")
            if pred == y:
                correct += 1
        accuracy = correct / len(X_test)
        print(f"Test Accuracy: {accuracy * 100:.2f}%")

X_train = [
    (0.6, 0.9), (0.2, 0.3), (0.8, 0.5), (1.0, 1.0), (0.0, 0.0),
    (0.9, 0.1), (0.4, 0.4), (0.3, 0.8), (0.7, 0.2), (0.1, 0.9)
]
Y_train = [1, 0, 1, 1, 0, 1, 0, 1, 1, 0]

# Separate test set
X_test = [(0.5, 0.5), (0.0, 1.0), (1.0, 0.0)]
Y_test = [1, 0, 1]

model = TwoLayerNeuralNetwork(learning_rate=0.1)
model.train(X_train, Y_train, epochs=10000)

model.test(X_test, Y_test)

Epoch 1000, Loss: 6.5748
Epoch 2000, Loss: 0.1658
Epoch 3000, Loss: 0.0472
Epoch 4000, Loss: 0.0267
Epoch 5000, Loss: 0.0184
Epoch 6000, Loss: 0.0140
Epoch 7000, Loss: 0.0112
Epoch 8000, Loss: 0.0094
Epoch 9000, Loss: 0.0080
Epoch 10000, Loss: 0.0070
Input: (0.5, 0.5) → Predicted: 1, Actual: 1
Input: (0.0, 1.0) → Predicted: 0, Actual: 0
Input: (1.0, 0.0) → Predicted: 1, Actual: 1
Test Accuracy: 100.00%
